# Data Preprocessing

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.preprocessing import StandardScaler
from IPython.display import display

## 1. File Paths

In [2]:
raw_data_path = Path('../data/raw/winequality-red.csv')
processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

## 2. Load Dataset

In [3]:
df = pd.read_csv(raw_data_path)
print(f'Original shape: {df.shape}')
display(df.head())

Original shape: (1599, 12)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


## 3. Preprocessing

In [4]:
missing_values = df.isnull().sum().sum()
duplicate_rows = df.duplicated().sum()

print(f'Total missing values: {missing_values}')
print(f'Duplicate rows before cleaning: {duplicate_rows}')

Total missing values: 0
Duplicate rows before cleaning: 240


## 4. Remove Duplicate Rows

In [5]:
df_no_duplicates = df.drop_duplicates().reset_index(drop=True)

print(f'Shape after duplicate removal: {df_no_duplicates.shape}')
print(f'Duplicate rows after cleaning: {df_no_duplicates.duplicated().sum()}')

Shape after duplicate removal: (1359, 12)
Duplicate rows after cleaning: 0


## 5. Outlier Removal

In [6]:
feature_columns = [col for col in df_no_duplicates.columns if col != 'quality']

Q1 = df_no_duplicates[feature_columns].quantile(0.25)
Q3 = df_no_duplicates[feature_columns].quantile(0.75)
IQR = Q3 - Q1

mask = ~((df_no_duplicates[feature_columns] < (Q1 - 1.5 * IQR)) | (df_no_duplicates[feature_columns] > (Q3 + 1.5 * IQR))).any(axis=1)
df_clean = df_no_duplicates[mask].reset_index(drop=True)

print(f'Shape before outlier removal: {df_no_duplicates.shape}')
print(f'Shape after outlier removal: {df_clean.shape}')
print(f'Rows removed as outliers: {df_no_duplicates.shape[0] - df_clean.shape[0]}')
display(df_clean.head())

Shape before outlier removal: (1359, 12)
Shape after outlier removal: (1019, 12)
Rows removed as outliers: 340


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5


## 6. Feature Scaling
`StandardScaler` 

In [7]:
X = df_clean.drop(columns='quality')
y = df_clean['quality']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

df_scaled = pd.DataFrame(X_scaled, columns=X.columns)
df_scaled['quality'] = y.values

display(df_scaled.head())

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,-0.522178,1.061042,-1.376357,-0.661123,-0.163865,-0.448210,-0.315211,0.775525,1.415708,-0.620504,-0.994033,5
1,-0.250872,2.137073,-1.376357,0.887576,1.306135,1.136466,0.924797,0.156543,-0.923491,0.424753,-0.590761,5
2,-0.250872,1.419719,-1.156846,0.223848,0.905226,0.004554,0.436309,0.280340,-0.470743,0.163439,-0.590761,5
3,2.055231,-1.449696,1.696791,-0.661123,-0.230683,0.230937,0.661765,0.899321,-1.225323,-0.446294,-0.590761,6
4,-0.522178,0.821924,-1.376357,-0.882366,-0.230683,-0.221828,-0.089755,0.775525,1.415708,-0.620504,-0.994033,5


## 7. Save Processed Datasets

- `winequality_cleaned.csv` 
- `winequality_scaled.csv` 

In [8]:
cleaned_output_path = processed_dir / 'winequality_cleaned.csv'
scaled_output_path = processed_dir / 'winequality_scaled.csv'

df_clean.to_csv(cleaned_output_path, index=False)
df_scaled.to_csv(scaled_output_path, index=False)

print(f'Cleaned dataset saved to: {cleaned_output_path}')
print(f'Scaled dataset saved to: {scaled_output_path}')

Cleaned dataset saved to: ..\data\processed\winequality_cleaned.csv
Scaled dataset saved to: ..\data\processed\winequality_scaled.csv
